In [1]:
import plotly.express as px
import pandas as pd

In [2]:
RESULTS_FILES = [
    "../results/sc-runs/runs-synth-1789418869.csv",
    "../results/sc-runs/runs-ev-1789672175.csv",
]

SCENARIOS = [
    "uniform_everywhere",
    "uniform_halflowestscalars",
    "uniform_lowestrowsumsonly",
    "uniform_randomrows"
]

LOWSC = SCENARIOS[1:]

LINEPLOT_SIZE = {"width": 1000, "height": 800}
DISTLABELS = {
    "nandist_euclidean": "nandist Euclidean",
    "dixon_pds_sqeuclidean": "PDS",
    "sqeuclidean": "CC sqeuclidean",
    "eirola_esd_gmm": "ESD/GMM",
    "eirola_esd_mvn": "ESD/MVN",
    "mesquita_eed": "EED",
}
SCENARIO_LABELS = {
    "uniform_everywhere": "All rows",
    "uniform_halflowestscalars": "Lower half of z-values",
    "uniform_lowestrowsumsonly": "30% of rows with lowest sum",
    "uniform_randomrows": "30% of rows, randomly selected",
}

In [3]:
RESULTS_FILE = RESULTS_FILES[1]

In [4]:
df = pd.read_csv(RESULTS_FILE)
df

,Unnamed: 0,Distance,Run,Missingness,Replicate,CCC,Rand,aRand
0,0,mesquita_eed,uniform_halflowestscalars,70,49,0.888985,1.0,1.0
1,1,mesquita_eed,uniform_halflowestscalars,70,48,0.886649,1.0,1.0
2,2,mesquita_eed,uniform_halflowestscalars,70,47,0.888689,1.0,1.0
3,3,mesquita_eed,uniform_halflowestscalars,70,46,0.889706,1.0,1.0
4,4,mesquita_eed,uniform_halflowestscalars,70,45,0.889364,1.0,1.0
...,...,...,...,...,...,...,...,...
19219,19219,sqeuclidean,uniform_everywhere,1,3,0.849404,1.0,1.0
19220,19220,sqeuclidean,uniform_everywhere,1,2,0.849592,1.0,1.0
19221,19221,sqeuclidean,uniform_everywhere,1,1,0.849948,1.0,1.0
19222,19222,sqeuclidean,uniform_everywhere,1,0,0.849836,1.0,1.0


In [5]:
missigness_vals = sorted(list(set(df["Missingness"])))
distances = sorted(list(set(df["Distance"])))

In [6]:
vals = []

for r in SCENARIOS:
    rfiltered_df = df[df["Run"] == r]
    for m in missigness_vals:
        mfiltered_df = rfiltered_df[rfiltered_df["Missingness"] == m]
        for d in distances:
            dfiltered_df = mfiltered_df[mfiltered_df["Distance"] == d][["CCC", "aRand"]]
            var = dfiltered_df.var(axis=0, numeric_only=True)
            median = dfiltered_df.median(axis=0, numeric_only=True)
            vals.append([r, d, m, median["CCC"], var["CCC"], median["aRand"], var["aRand"]])

condensed_df = pd.DataFrame(
    columns=["Scenario", "Distance", "Missingness in %", "Median CCC", "CCC variance", "Median ARI", "ARI variance"],
    data=vals
)

## CCC

In [7]:
# Median CCC
fig = px.line(
    condensed_df,
    x="Missingness in %",
    y="Median CCC",
    color="Distance",
    range_y=[0.0, 0.86],
    facet_col="Scenario",
    facet_col_wrap=2,
    title=""
)
fig.update_layout(**LINEPLOT_SIZE)
fig.for_each_annotation(lambda a: a.update(text=SCENARIO_LABELS[a.text.split("=")[-1]]))
fig.for_each_trace(lambda a: a.update(name=DISTLABELS[a.name]))
fig.show()

In [8]:
# Median CCC, highlighted scenarios
fig = px.line(
    condensed_df[condensed_df["Scenario"].isin(LOWSC)],
    x="Missingness in %",
    y="Median CCC",
    color="Distance",
    facet_row="Scenario",
    title=""
)
fig.update_layout(**LINEPLOT_SIZE)
fig.for_each_annotation(lambda a: a.update(text=SCENARIO_LABELS[a.text.split("=")[-1]]))
fig.for_each_trace(lambda a: a.update(name=DISTLABELS[a.name]))
fig.show()

In [9]:
# CCC variance
fig = px.line(
    condensed_df,
    x="Missingness in %",
    y="CCC variance",
    color="Distance",
    facet_col="Scenario",
    facet_col_wrap=2,
    title=""
)
fig.update_layout(**LINEPLOT_SIZE)
fig.for_each_annotation(lambda a: a.update(text=SCENARIO_LABELS[a.text.split("=")[-1]]))
fig.for_each_trace(lambda a: a.update(name=DISTLABELS[a.name]))
fig.show()

In [10]:
# CCC variance, hightlighted scenarios
fig = px.line(
    condensed_df[condensed_df["Scenario"].isin(LOWSC)],
    x="Missingness in %",
    y="CCC variance",
    color="Distance",
    facet_row="Scenario",
    title=""
)
fig.update_layout(**LINEPLOT_SIZE)
fig.for_each_annotation(lambda a: a.update(text=SCENARIO_LABELS[a.text.split("=")[-1]]))
fig.for_each_trace(lambda a: a.update(name=DISTLABELS[a.name]))
fig.show()

## aRand

In [11]:
# Median aRand
fig = px.line(
    condensed_df,
    x="Missingness in %",
    y="Median ARI",
    color="Distance",
    facet_col="Scenario",
    facet_col_wrap=2,
    title=""
)
fig.update_layout(**LINEPLOT_SIZE)
fig.for_each_annotation(lambda a: a.update(text=SCENARIO_LABELS[a.text.split("=")[-1]]))
fig.for_each_trace(lambda a: a.update(name=DISTLABELS[a.name]))
fig.show()

In [12]:
# aRand variance
fig = px.line(
    condensed_df,
    x="Missingness in %",
    y="ARI variance",
    color="Distance",
    facet_col="Scenario",
    facet_col_wrap=2,
    title=""
)
fig.update_layout(**LINEPLOT_SIZE)
fig.for_each_annotation(lambda a: a.update(text=SCENARIO_LABELS[a.text.split("=")[-1]]))
fig.for_each_trace(lambda a: a.update(name=DISTLABELS[a.name]))
fig.show()